# Frat Chore — Skip Prediction ML

This notebook trains a Random Forest classifier to predict **will a member skip their chore this week?**

Sections:
1. Setup & BigQuery connection
2. Exploratory analysis
3. Feature engineering
4. Model training & evaluation
5. Risk scores
6. Smart assignment suggestions

**Requirements:** Run this notebook in Google Colab with your GCP project credentials.

## 1. Setup & BigQuery Connection

In [ ]:
# Install dependencies (run once in Colab)
!pip install google-cloud-bigquery db-dtypes pandas-gbq --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# ---- Authenticate with Google Cloud ----
from google.colab import auth
auth.authenticate_user()
print('Authenticated!')

In [ ]:
from google.cloud import bigquery

# ---- REPLACE THIS WITH YOUR PROJECT ID ----
PROJECT_ID = 'YOUR_GCP_PROJECT_ID'
DATASET    = 'frat_chores'

client = bigquery.Client(project=PROJECT_ID)

def bq_query(sql):
    return client.query(sql).to_dataframe()

print(f'Connected to BigQuery project: {PROJECT_ID}')

In [ ]:
# Load all tables
submissions = bq_query(f"""
    SELECT * FROM `{PROJECT_ID}.{DATASET}.chore_submissions`
""")

fines = bq_query(f"""
    SELECT * FROM `{PROJECT_ID}.{DATASET}.chore_fines`
""")

assignments = bq_query(f"""
    SELECT * FROM `{PROJECT_ID}.{DATASET}.chore_assignments`
""")

print(f'Assignments: {len(assignments):,} rows')
print(f'Submissions: {len(submissions):,} rows')
print(f'Fines:       {len(fines):,} rows')

# Preview
assignments.head(3)

## 2. Exploratory Analysis

In [ ]:
# ---- Build master dataframe: one row per (member, chore, week, semester) ----
# Mark each assignment as skipped or not

# Normalize
submissions['submitted_at'] = pd.to_datetime(submissions['submitted_at'], errors='coerce')
assignments['assigned_date'] = pd.to_datetime(assignments['assigned_date'], errors='coerce')

# Which (member, chore, week_start) combos resulted in a verified/passed submission?
passed = submissions[
    (submissions['auto_status'] == 'passed') | (submissions['human_status'] == 'verified')
][['member_id', 'chore_name', 'week_start', 'semester']].drop_duplicates()
passed['submitted'] = True

# Merge with assignments
# Each assignment row is one member-chore pair per semester
# We need to cross with weeks — but BQ only stores the assignment, not week-level
# So we use fines as a proxy for skips (fined = skipped that week)
fines_key = fines[['member_id', 'chore_name', 'week_start', 'semester']].drop_duplicates()
fines_key['skipped'] = True

# Build base: fines are confirmed skips; submissions are confirmed completions
base = pd.concat([
    fines_key.assign(skipped=1),
    passed.rename(columns={'submitted': 'skipped'}).assign(skipped=0)
]).drop_duplicates(subset=['member_id', 'chore_name', 'week_start', 'semester'], keep='last')

print(f'Total records: {len(base):,}')
print(f'Skip rate: {base["skipped"].mean():.1%}')
base.head()

In [ ]:
# ---- Compliance rate per member ----
member_compliance = (
    base.groupby('member_id')['skipped']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'skips', 'count': 'total'})
)
member_compliance['compliance_rate'] = 1 - member_compliance['skips'] / member_compliance['total']

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(member_compliance['compliance_rate'], bins=20, color='#003087', edgecolor='#FFD700', linewidth=0.5)
ax.set_title('Member Compliance Rate Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Compliance Rate (1.0 = perfect)')
ax.set_ylabel('Number of Members')
ax.axvline(member_compliance['compliance_rate'].mean(), color='#FFD700', linestyle='--', label='Mean')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mean compliance: {member_compliance["compliance_rate"].mean():.1%}')
print(f'Bottom 10 members:')
member_compliance.sort_values('compliance_rate').head(10)

In [ ]:
# ---- Compliance rate per chore ----
chore_compliance = (
    base.groupby('chore_name')['skipped']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'skips', 'count': 'total'})
)
chore_compliance['skip_rate'] = chore_compliance['skips'] / chore_compliance['total']
chore_compliance = chore_compliance.sort_values('skip_rate', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#dc2626' if r > 0.3 else '#ca8a04' if r > 0.15 else '#16a34a'
          for r in chore_compliance['skip_rate']]
ax.barh(chore_compliance.index, chore_compliance['skip_rate'] * 100, color=colors)
ax.set_title('Chore Skip Rate (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Skip Rate (%)')
ax.axvline(chore_compliance['skip_rate'].mean() * 100, color='#003087', linestyle='--', alpha=0.7, label='Average')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---- Heatmap: member vs chore skip rate ----
# Pivot: rows = members, cols = chores, values = skip_rate
pivot = base.pivot_table(
    index='member_id', columns='chore_name', values='skipped', aggfunc='mean'
)

# Only show members and chores with enough data
min_obs = 2
pivot_filtered = pivot.loc[
    pivot.notna().sum(axis=1) >= min_obs,
    pivot.notna().sum(axis=0) >= min_obs
]

if not pivot_filtered.empty:
    fig, ax = plt.subplots(figsize=(max(8, pivot_filtered.shape[1] * 0.5),
                                     max(6, pivot_filtered.shape[0] * 0.4)))
    sns.heatmap(
        pivot_filtered.fillna(0.5),
        cmap='RdYlGn_r', vmin=0, vmax=1,
        ax=ax, linewidths=0.2, linecolor='#e2e8f0',
        cbar_kws={'label': 'Skip Rate (1 = always skips)'}
    )
    ax.set_title('Member × Chore Skip Rate Heatmap', fontsize=13, fontweight='bold')
    ax.set_xlabel('Chore')
    ax.set_ylabel('Member ID')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print('Not enough data for heatmap yet — needs at least 2 semesters of history.')

In [ ]:
# ---- Week-of-semester skip rate curve ----
# Parse week_start to get week number within semester

def week_number_in_semester(df, semester_col='semester', week_col='week_start'):
    """Assign week 1..N within each semester."""
    df = df.copy()
    df[week_col] = pd.to_datetime(df[week_col], errors='coerce')
    df['week_num'] = df.groupby(semester_col)[week_col].rank(method='dense').astype(int)
    return df

if 'week_start' in base.columns:
    base_w = week_number_in_semester(base)
    weekly_skip = base_w.groupby('week_num')['skipped'].mean()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(weekly_skip.index, weekly_skip.values * 100,
            marker='o', color='#003087', linewidth=2, markersize=6)
    ax.fill_between(weekly_skip.index, weekly_skip.values * 100, alpha=0.2, color='#003087')
    ax.set_title('Skip Rate by Week of Semester (does laziness peak mid-semester?)',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Week of Semester')
    ax.set_ylabel('Skip Rate (%)')
    ax.set_ylim(0, 100)
    plt.tight_layout()
    plt.show()

## 3. Feature Engineering

In [ ]:
# ---- Build feature matrix ----------------------------------
# Target: skipped (1) or completed (0)

# Chore group sizes from config
CHORE_SIZES = {
    'Monday Lunch Setup': 2, 'Monday Lunch Cleanup': 1, 'Monday Dinner Setup': 2,
    'Monday Dinner Cleanup': 3, 'Tuesday Lunch Setup': 2, 'Tuesday Lunch Cleanup': 1,
    'Tuesday Dinner Setup': 2, 'Tuesday Dinner Cleanup': 2, 'Wednesday Lunch Setup': 2,
    'Wednesday Lunch Cleanup': 2, 'Wednesday Dinner Setup': 2, 'Wednesday Dinner Cleanup': 1,
    'Thursday Lunch Setup': 2, 'Thursday Lunch Cleanup': 1, 'Thursday Dinner Setup': 1,
    'Thursday Dinner Cleanup': 1, 'Friday Lunch Setup': 2, 'Friday Lunch Cleanup': 1,
    'Monday Mail': 1, 'Wednesday Mail': 1, 'Friday Mail': 1,
    'Chapter Setup/Cleanup': 4, 'Living Room/Chapter Room Cleanup': 3,
    'Basement': 2, 'Brotherhood Kitchen': 2, '1F Restrooms': 2,
    '2F Restrooms': 2, '3F Restrooms': 2, '3rd Floor': 2, 'Laundry Room': 2, 'Outside': 2,
}

# Annotated exam weeks (update each semester!)
# Format: 'YYYY-MM-DD' of the Monday of exam week
EXAM_WEEKS = set([])

# ---- Compute historical compliance per member (rolling) ----
base_sorted = base_w.sort_values(['member_id', 'semester', 'week_num'])

def rolling_compliance(group):
    """Expanding mean of (1-skipped) before the current row."""
    return (1 - group['skipped']).expanding().mean().shift(1)  # shift: no data leakage

base_sorted['member_hist_compliance'] = (
    base_sorted.groupby('member_id', group_keys=False)
    .apply(rolling_compliance)
)

# ---- Compute chore historical skip rate ----
chore_skip_map = chore_compliance['skip_rate'].to_dict()
base_sorted['chore_hist_skip_rate'] = base_sorted['chore_name'].map(chore_skip_map).fillna(0.15)

# ---- Days since last fine ----
if 'week_start' in base_sorted.columns and 'issued_at' in fines.columns:
    fines_ts = fines.copy()
    fines_ts['issued_at'] = pd.to_datetime(fines_ts['issued_at'], errors='coerce')
    last_fine = fines_ts.groupby('member_id')['issued_at'].max().reset_index()
    last_fine.columns = ['member_id', 'last_fine_date']
    base_sorted = base_sorted.merge(last_fine, on='member_id', how='left')
    base_sorted['week_start_dt'] = pd.to_datetime(base_sorted['week_start'], errors='coerce')
    base_sorted['days_since_fine'] = (
        (base_sorted['week_start_dt'] - base_sorted['last_fine_date']).dt.days
    ).clip(0, 365).fillna(365)
else:
    base_sorted['days_since_fine'] = 365  # never fined = safe default

# ---- Encode chore type ----
le_chore = LabelEncoder()
base_sorted['chore_encoded'] = le_chore.fit_transform(base_sorted['chore_name'])

# ---- Group size ----
base_sorted['group_size'] = base_sorted['chore_name'].map(CHORE_SIZES).fillna(2)

# ---- Exam week flag ----
base_sorted['is_exam_week'] = base_sorted['week_start'].isin(EXAM_WEEKS).astype(int)

# ---- Final feature set ----
FEATURES = [
    'week_num',
    'chore_encoded',
    'member_hist_compliance',
    'chore_hist_skip_rate',
    'is_exam_week',
    'group_size',
    'days_since_fine',
]
TARGET = 'skipped'

df_model = base_sorted[FEATURES + [TARGET, 'semester']].dropna()
print(f'Model dataset: {len(df_model):,} rows')
print(f'Skip rate: {df_model[TARGET].mean():.1%}')
df_model[FEATURES].describe()

## 4. Skip Prediction Model

In [ ]:
# ---- Train/test split by semester --------------------------
# Train on all semesters except the most recent; test on most recent

semesters = sorted(df_model['semester'].unique())
print('Semesters:', semesters)

if len(semesters) < 2:
    print('WARNING: Only 1 semester of data. Using 80/20 random split instead.')
    from sklearn.model_selection import train_test_split
    train_df, test_df = train_test_split(df_model, test_size=0.2, random_state=42, stratify=df_model[TARGET])
else:
    test_sem  = semesters[-1]
    train_df  = df_model[df_model['semester'] != test_sem]
    test_df   = df_model[df_model['semester'] == test_sem]
    print(f'Train semesters: {semesters[:-1]}')
    print(f'Test semester:   {test_sem}')

X_train = train_df[FEATURES]
y_train = train_df[TARGET]
X_test  = test_df[FEATURES]
y_test  = test_df[TARGET]

print(f'Train: {len(X_train):,} rows | Test: {len(X_test):,} rows')

In [ ]:
# ---- Train Random Forest -----------------------------------
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=5,
    class_weight='balanced',  # handle imbalanced skip rate
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Cross-val on training set
cv_scores = cross_val_score(rf, X_train, y_train, cv=5, scoring='roc_auc')
print(f'CV ROC-AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

In [ ]:
# ---- Evaluation --------------------------------------------
from sklearn.metrics import roc_auc_score

y_pred     = rf.predict(X_test)
y_prob     = rf.predict_proba(X_test)[:, 1]

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Completed', 'Skipped']))

print(f'ROC-AUC (test): {roc_auc_score(y_test, y_prob):.3f}')

# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Completed', 'Skipped'],
    colorbar=False, ax=ax,
    cmap='Blues'
)
ax.set_title('Confusion Matrix (Test Set)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Feature importances -----------------------------------
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(importances.index, importances.values * 100, color='#003087', edgecolor='#FFD700', linewidth=0.5)
ax.set_title('Feature Importances — What drives skipping?', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance (%)')
for bar, val in zip(bars, importances.values * 100):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Save model --------------------------------------------
import pickle
from datetime import date

model_path = f'skip_model_{date.today().strftime("%Y%m%d")}.pkl'
with open(model_path, 'wb') as f:
    pickle.dump({'model': rf, 'features': FEATURES, 'chore_encoder': le_chore}, f)

print(f'Model saved to: {model_path}')

## 5. Risk Scores for Current Semester

In [ ]:
# ---- Per-member risk score for the CURRENT semester --------
current_semester = semesters[-1]

current = df_model[df_model['semester'] == current_semester].copy()

if len(current) > 0:
    current['skip_prob'] = rf.predict_proba(current[FEATURES])[:, 1]

    member_risk = (
        current.groupby('member_id')['skip_prob']
        .mean()
        .reset_index()
        .rename(columns={'skip_prob': 'avg_skip_prob'})
        .sort_values('avg_skip_prob', ascending=False)
    )

    print(f'=== High-Risk Members — {current_semester} ===')
    print(member_risk.head(10).to_string(index=False))

    # Visualize
    fig, ax = plt.subplots(figsize=(10, 5))
    top20 = member_risk.head(20)
    colors = ['#dc2626' if p > 0.6 else '#ca8a04' if p > 0.35 else '#16a34a'
              for p in top20['avg_skip_prob']]
    ax.bar(range(len(top20)), top20['avg_skip_prob'] * 100, color=colors)
    ax.set_xticks(range(len(top20)))
    ax.set_xticklabels(top20['member_id'], rotation=45, ha='right', fontsize=8)
    ax.set_title(f'Top 20 Highest Risk Members — {current_semester}', fontweight='bold')
    ax.set_ylabel('Predicted Skip Probability (%)')
    ax.axhline(50, color='black', linestyle='--', alpha=0.4, label='50% threshold')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No current semester data available. Run after some weeks have passed.')

In [ ]:
# ---- Per-chore risk score ----------------------------------
if len(current) > 0:
    chore_risk = (
        current.groupby('chore_name')['skip_prob']
        .mean()
        .reset_index()
        .rename(columns={'skip_prob': 'avg_skip_prob'})
        .sort_values('avg_skip_prob', ascending=True)
    )

    fig, ax = plt.subplots(figsize=(10, 8))
    colors = ['#dc2626' if p > 0.5 else '#ca8a04' if p > 0.25 else '#16a34a'
              for p in chore_risk['avg_skip_prob']]
    ax.barh(chore_risk['chore_name'], chore_risk['avg_skip_prob'] * 100, color=colors)
    ax.set_title(f'Chore Risk Ranking — {current_semester}', fontweight='bold')
    ax.set_xlabel('Predicted Skip Probability (%)')
    plt.tight_layout()
    plt.show()

## 6. Smart Assignment Suggestions

In [ ]:
# ---- Pair high-risk members with low-risk members ----------
# Strategy: for each chore with group_size >= 2,
# pair at least one low-risk member with each high-risk member.

if len(current) > 0:
    HIGH_RISK_THRESHOLD = 0.50
    LOW_RISK_THRESHOLD  = 0.20

    high_risk_members = member_risk[member_risk['avg_skip_prob'] >= HIGH_RISK_THRESHOLD]['member_id'].tolist()
    low_risk_members  = member_risk[member_risk['avg_skip_prob'] <= LOW_RISK_THRESHOLD]['member_id'].tolist()

    print(f'High-risk members (skip prob ≥ {HIGH_RISK_THRESHOLD:.0%}): {len(high_risk_members)}')
    print(f'Low-risk members  (skip prob ≤ {LOW_RISK_THRESHOLD:.0%}):  {len(low_risk_members)}')

    # Generate suggestions: chores needing 2+ people → include at least 1 low-risk
    chore_suggestions = []
    for chore, size in CHORE_SIZES.items():
        if size < 2:
            continue
        chore_risk_val = chore_risk[chore_risk['chore_name'] == chore]['avg_skip_prob']
        if chore_risk_val.empty:
            continue
        if chore_risk_val.values[0] > HIGH_RISK_THRESHOLD:
            chore_suggestions.append({
                'chore': chore,
                'group_size': size,
                'chore_skip_prob': round(chore_risk_val.values[0], 3),
                'recommendation': f'Assign ≥1 low-risk member (out of {size} total)'
            })

    suggestions_df = pd.DataFrame(chore_suggestions).sort_values('chore_skip_prob', ascending=False)
    print(f'\n=== Chores needing a reliable anchor member ===')
    print(suggestions_df.to_string(index=False))

    # Export CSV for house manager
    suggestions_df.to_csv('smart_assignments.csv', index=False)
    member_risk.to_csv('member_risk_scores.csv', index=False)
    chore_risk.to_csv('chore_risk_scores.csv', index=False)
    print('\nCSVs saved: smart_assignments.csv, member_risk_scores.csv, chore_risk_scores.csv')

In [ ]:
# ---- Download files to local machine (Colab only) ----------
from google.colab import files

for fname in ['smart_assignments.csv', 'member_risk_scores.csv', 'chore_risk_scores.csv', model_path]:
    try:
        files.download(fname)
    except Exception as e:
        print(f'Could not download {fname}: {e}')